# Convolutional Neural Networks

Convolutional Neural Networks (CNNs) sind die Standardarchitektur für die Arbeit mit Bildern. Ein einfaches vollständig verbundenes Netz behandelt jedes Pixel als unabhängige Eingabe. Damit geht verloren, dass benachbarte Pixel zusammengehören, und das Modell muss eine riesige Zahl an Gewichten lernen. Ein CNN schiebt stattdessen kleine lernbare Filter über das Bild. So kann es ein nützliches Muster (eine Kante, eine Ecke, eine Textur) erkennen, wo auch immer es auftaucht, und braucht dafür weit weniger Parameter.

Dieses Notebook baut ein intuitives Verständnis der zentralen Operationen (Faltung, Stride, Padding, Pooling) auf und trainiert dann mit Keras ein kleines CNN auf dem MNIST-Datensatz mit Ziffern.

## Lernziele

Am Ende dieses Notebooks kannst du:

- erklären, wie Faltung, Stride, Padding und Pooling ein Bild transformieren.
- in Keras ein Convolutional Neural Network mit `Conv2D`-, `MaxPooling2D`-, `Flatten`- und `Dense`-Schichten erstellen.
- das Modell auf MNIST trainieren und seine Accuracy- und Verlustkurven lesen, um Overfitting zu erkennen.
- das trainierte Modell mit Vorhersagen und einer Confusion Matrix evaluieren.

## Wie eine Faltung funktioniert

Eine Faltung schiebt eine kleine Matrix aus Gewichten (einen **Kernel** oder **Filter**) über das Bild. Der Kernel ist viel kleiner als das Bild, also werden an jeder Position dieselben Gewichte wiederverwendet. Genau diese gemeinsame Nutzung der Gewichte macht CNNs effizient: Ein Filter, der eine senkrechte Kante erkennt, ist überall im Bild nützlich, also brauchen wir keinen eigenen Satz Gewichte pro Pixel.

### Faltung

So faltest du einen Kernel mit einem Bild:

- Schieb den Kernel (eine kleine Matrix aus Gewichten) über das 2D-Bild.
- Multipliziere an jeder Position jedes Gewicht des Kernels mit dem Pixel, das es überdeckt (elementweise).
- Summiere diese Produkte zu einem einzigen Ausgabepixel.

<img src="https://miro.medium.com/max/1400/1*Fw-ehcNBR9byHtho-Rxbtw.gif" alt="Drawing" style="width: 400px;"/>

### Stride

Der **Stride** ist die Schrittweite, um die sich der Kernel zwischen zwei Positionen weiterbewegt. Bei einem Stride von 1 rückt der Kernel jeweils ein Pixel weiter, bei einem Stride von 2 überspringt er jede zweite Position (dadurch wird auch die Ausgabe kleiner). Die Animation unten zeigt einen Stride von 2.

<img src="https://miro.medium.com/max/588/1*BMngs93_rm2_BpJFH2mS0Q.gif" alt="Drawing" style="width: 300px;"/>

Wir falten den Filter mit dem Bild, indem wir den 3x3-Filter über das 6x6-Bild schieben. An jeder Position multiplizieren wir die überlappenden Pixel, summieren sie und speichern die Summe als Ausgabepixel für diese Position. Dann schieben wir den Filter eine Zelle nach rechts und wiederholen das. Ist die Zeile fertig, gehen wir eine Zeile nach unten, und so weiter.

Berechnen wir ein paar Werte gemeinsam, danach machst du den Rest selbst.

![](assets/Convolution.gif)

Mach jetzt selbst weiter, bis der Filter das ganze Originalbild abgedeckt hat.

Das Ergebnis kannst du in `assets/convolved.jpeg` prüfen.

**F**: Welche Größe hat das resultierende Bild, wenn das Originalbild `n x n` groß ist und der Filter `f x f`?

**A**: `n-f+1`

Du hast gerade deinen ersten Algorithmus zur Kantenerkennung ausgeführt.

So funktionieren, ganz vereinfacht gesagt, Convolutional Neural Networks. Der Unterschied ist, dass wir die Werte der Filter nicht von Hand wählen: Die Filter sind die Parameter (die Gewichte, deine `w`s), die das Netz während des Trainings lernt.

### Padding

**Padding** fügt vor der Faltung zusätzliche Pixel an den Rändern des Bildes hinzu. Wir haben gesehen, dass die Ausgabe kleiner ist als die Eingabe und dass Randpixel seltener vom Filter erfasst werden als Pixel in der Mitte, was nicht immer erwünscht ist. Ein Padding, bei dem die Ausgabe ihre Größe behält, löst beide Probleme.

<img src="https://miro.medium.com/max/790/1*1okwhewf5KCtIPaFib4XaA.gif" alt="Drawing" style="width: 300px;"/>

Wie viel Padding sollen wir verwenden? Zwei gängige Optionen:

- `valid`: kein Padding.
- `same`: so auffüllen, dass die Ausgabe dieselbe Größe behält wie die Eingabe.

**F**: Welche Größe hat das resultierende Bild, wenn das Originalbild `n x n` groß ist, der Filter `f x f` und wir auf jeder Seite `p` Pixel Padding verwenden?

**A**: `n-f+1+2p`

**F**: Welche Größe hat das resultierende Bild, wenn das Originalbild `n x n` groß ist, der Filter `f x f` und wir ein Padding von `p` und einen Stride von `s` verwenden?

**A**: `(n-f+2p)/s + 1`

### Pooling

Neben Faltungsschichten verwenden CNNs **Pooling-Schichten**. Pooling fasst die Features, die eine Faltungsschicht erzeugt, zusammen (Pooling) und verkleinert die räumliche Größe der Daten.

**Max Pooling**, die gängigste Methode, behält in jedem Pooling-Fenster nur den höchsten Wert.

![max_pooling.png](assets/max_pooling.png)

Eine weitere Option ist **Average Pooling**, das statt des Maximums den Mittelwert jedes Fensters behält.

### Farbbilder (RGB)

Bei einem Farbbild (RGB) beginnen wir mit drei Filtern, einem pro Farbkanal, falten jeden Kanal wie zuvor und summieren dann die drei Ergebnisse zu einem einzigen Ausgabebild (hier 4x4x1).

### Eine Schicht eines CNN

Eine CNN-Schicht wendet meist mehr als einen Filter an. Die dritte Dimension der Ausgabe entspricht der Anzahl der verwendeten Filter, jeder Filter erzeugt also seine eigene Feature Map.

![rgb.png](assets/rgb.png)

In CNNs verwenden wir in der Regel mehr als einen Filter pro Schicht, daher entspricht die Tiefe der Ausgabe der Anzahl der Filter.

![rgb_2.png](assets/rgb_2.png)

In CNN-Diagrammen wird das meist so gezeichnet:

![one_layer.png](assets/one_layer.png)

### Auffrischung: Bild-Kernel

Beschäftige dich eine Weile mit dieser interaktiven Erklärung der [Bild-Kernel](https://setosa.io/ev/image-kernels/) (probier ruhig alle Einstellungen aus) und diskutiere:

- Was bedeutet die Zahl an jeder Pixelposition in einem Graustufenbild?
- Was ist ein Kernel?
- Wie wird ein Kernel auf ein Bild angewendet?
- Wie verändert sich das Ausgabebild, wenn sich die Zahlen im Kernel ändern?
- Welche Features eines Bildes können wir mit einem Kernel hervorheben?

### Beispiel für eine CNN-Architektur

Ein typisches CNN stapelt einige Faltungsschichten (oft im Wechsel mit Pooling-Schichten), flacht dann das Ergebnis ab und endet mit einer oder mehreren **vollständig verbundenen** (dichten) Schichten, die die abschließende Klassifikation übernehmen. Unten abgebildet ist eine der klassischen Architekturen, LeNet-5.

Das Diagramm unten zeigt dieselbe Idee als Ablauf: Das Bild durchläuft abwechselnd Faltungs- und Pooling-Schichten, die Features extrahieren, wird zu einem Vektor abgeflacht und von dichten Schichten klassifiziert.

```mermaid
flowchart LR
    A[Eingabebild] --> B[Conv + ReLU]
    B --> C[Max Pooling]
    C --> D[Conv + ReLU]
    D --> E[Max Pooling]
    E --> F[Flatten]
    F --> G[Dichte Schichten]
    G --> H[Klassenwahrscheinlichkeiten]
```

![cnn_with_pooling.png](assets/cnn_with_pooling.png)

Die Verwendung von Filtern hat zwei Vorteile:

- Sie senkt die Anzahl der Parameter, weil derselbe Filter im ganzen Bild wiederverwendet wird.
- Sie ist sinnvoll, weil ein Filter, der ein bestimmtes Feature erkennt, in vielen Bereichen eines Bildes nützlich ist.

Benachbarte Pixel hängen zusammen, und das ist wichtig:

- Würden wir das Bild sofort abflachen, ginge diese räumliche Information verloren.
- Weit entfernte Pixel hängen meist nicht zusammen, also müssen wir nicht jedes Eingabepixel mit jedem Ausgabepixel verbinden.

### Ein CNN visualisieren

Erkunde diese interaktive 3D-Visualisierung eines CNN, das Ziffern klassifiziert: https://adamharley.com/nn_vis/cnn/2d.html

### Weitere Schichten gegen Overfitting

- **Batch-Normalisierung**: macht Netze schneller und stabiler, indem sie die Eingaben jeder Schicht neu zentriert und neu skaliert.
- **Dropout**: eine Regularisierungstechnik, die während des Trainings zufällig Neuronen verwirft, um komplexe Koadaptationen zu verhindern, und damit wie eine Mittelung über viele Netze wirkt.

## Ein CNN mit Keras erstellen

Jetzt setzen wir die Theorie in die Praxis um: Wir laden den MNIST-Datensatz mit handgeschriebenen Ziffern und trainieren ein kleines CNN, um die Ziffern 0 bis 9 zu klassifizieren.

In [ ]:
# Benötigte Module importieren
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, MaxPooling2D, Flatten

from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np

### 0. Datensatz laden

In [ ]:
# MNIST-Daten aus den Keras-Datensätzen in Trainings- und Testdaten aufteilen
(xtrain, ytrain), (xtest, ytest) = mnist.load_data()

Prüf die Formen der Arrays.

In [ ]:
xtrain.shape, xtest.shape, ytrain.shape, ytest.shape

Erklärung des Ergebnisses: Der Trainingsdatensatz enthält 60.000 Bilder und der Testdatensatz 10.000, jedes davon ein 28x28-Graustufenbild, mit einem Label pro Bild.

#### In die Form (Bilder, Breite, Höhe, Kanäle) bringen

Eine 2D-Faltung erwartet, dass jedes Bild drei Dimensionen hat: Breite, Höhe und Anzahl der Farbkanäle. Farbbilder haben 3 Kanäle (RGB); ein Graustufenbild hat 1, also fügen wir am Ende eine Kanal-Dimension hinzu.

In [ ]:
xtrain.reshape(60000, 28, 28, 1).shape

In [ ]:
xtrain = xtrain.reshape(60000, 28, 28, 1)
xtest = xtest.reshape(10000, 28, 28, 1)

MNIST hat 10 Klassen, die Ziffern 0 bis 9:

In [ ]:
# Im MNIST-Datensatz gibt es 10 Klassen zum Kategorisieren, d. h. die Zahlen von 0-9
np.unique(ytrain)

#### Labels per One-Hot Encoding kodieren

Die Labels sind einzelne Ganzzahlen (zum Beispiel `3`), die Softmax-Ausgabeschicht erzeugt aber eine Wahrscheinlichkeit pro Klasse. Deshalb wandeln wir jedes Label in einen Vektor mit 10 Elementen um, der an der Position der richtigen Klasse eine 1 hat.

In [ ]:
# Eine Kopie von ytest erstellen, mehr Details unter (https://docs.python.org/3/library/copy.html)
# Das tun wir, weil wir die ursprünglichen Labels behalten und zusätzlich eine Variable mit den kodierten Labels haben wollen
ytest_true = ytest.copy()

In [ ]:
to_categorical(ytest)  # z. B. wie kategoriale Variablen per One-Hot Encoding kodiert werden

In [ ]:
ytrain = to_categorical(ytrain)
ytest = to_categorical(ytest)

In [ ]:
ytest

### 1. Modell definieren

Wir bauen das Netz als `Sequential`-Stapel aus Schichten. Der Aufbau folgt der obigen Architektur im LeNet-Stil: zwei Blöcke aus Faltung und Pooling, die Features extrahieren, dann eine `Flatten`- und eine `Dense`-Schicht mit Softmax, um in 10 Ziffern zu klassifizieren.

- Eine `Conv2D`-Schicht mit 6 Filtern, ReLU-Aktivierung, dann `MaxPooling2D`.
- Eine `Conv2D`-Schicht mit 16 Filtern, ReLU-Aktivierung, dann `MaxPooling2D`.
- `Flatten`, um die Feature Maps in einen Vektor umzuwandeln.
- Eine `Dense`-Schicht mit 10 Neuronen und Softmax-Aktivierung, eine Ausgabe pro Klasse.

Wir verwenden `padding='same'`, damit die Faltungen die räumliche Größe behalten, und überlassen das Downsampling dem Pooling.

In [ ]:
# TensorFlow-Modell mit der obigen Definition erstellen
from tensorflow.keras import Input
from tensorflow.keras import backend as K

K.clear_session()  # Cache der Modellparameter leeren
model = Sequential(
    [
        Input(shape=(28, 28, 1)),
        Conv2D(
            filters=6,
            kernel_size=(3, 3),
            strides=(1, 1),
            padding="same",
            activation="relu",
        ),
        MaxPooling2D(pool_size=(2, 2), strides=2),
        Conv2D(
            filters=16,
            kernel_size=(3, 3),
            strides=(1, 1),
            padding="same",
            activation="relu",
        ),
        MaxPooling2D(pool_size=(2, 2), strides=2),
        Flatten(),
        Dense(units=10, activation="softmax"),
    ]
)

#### Referenz zu den Schichten

- [Conv2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D)
- [MaxPooling2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPooling2D)
- [Flatten](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten)
- [Dense](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)

Die Anzahl der Neuronen in der letzten `Dense`-Schicht ist die Anzahl der Klassen. Verwende für die binäre Klassifikation in der letzten Schicht eine `sigmoid`-Aktivierung; für die Mehrklassen-Klassifikation `softmax`.

Für die Zwischenschichten ist [ReLU](https://www.tensorflow.org/api_docs/python/tf/keras/activations/relu) die empfohlene Standardwahl, sowohl für mehrschichtige Perzeptren als auch für CNNs. Sieh dir die vollständige Liste der [Aktivierungsfunktionen](https://www.tensorflow.org/api_docs/python/tf/keras/activations) an.

- [Allgemeine Richtlinien zum Erstellen von CNNs und zum Tunen von Hyperparametern](https://medium.com/data-science/a-guide-to-an-efficient-way-to-build-neural-network-architectures-part-ii-hyper-parameter-42efca01e5d7)
- [Hyperparameter-Tuning mit Keras Tuner](https://www.tensorflow.org/tutorials/keras/keras_tuner)

Prüf die Zusammenfassung des Modells.

In [ ]:
model.summary()

Erklärung des Ergebnisses: Das Modell hat nur **8.790 trainierbare Parameter**. Das ist für einen Bildklassifikator winzig und zeigt, wie sich die gemeinsame Nutzung der Gewichte auszahlt: Die Faltungsschichten verwenden dieselben kleinen Filter im ganzen Bild wieder, statt für jedes Pixel ein eigenes Gewicht zu lernen.

### 2. Modell kompilieren

Das Kompilieren legt fest, wie Keras trainiert: welcher Optimierer die Gewichte aktualisiert, welcher Verlust minimiert werden soll und welche Metrik ausgegeben wird. Wir verwenden `categorical_crossentropy`, weil die Labels über 10 Klassen per One-Hot Encoding kodiert sind, und verfolgen `accuracy` nach.

In [ ]:
model.compile(
    optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"]
)

#### Referenz zur Kompilierung

- [Optimierer](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers): Der [Adam](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam)-Optimierer ist eine beliebte, wirksame Standardwahl, und eine niedrige Lernrate (zum Beispiel 0,001 oder 0,0001) kann die Leistung verbessern.
- [Verlustfunktionen](https://www.tensorflow.org/api_docs/python/tf/keras/losses): Verwende `binary_crossentropy` für die binäre Klassifikation, `categorical_crossentropy` für die Mehrklassen-Klassifikation und eine Fehlermetrik wie `mean_squared_error` für die Regression.
- [Metriken](https://www.tensorflow.org/api_docs/python/tf/keras/metrics).

### 3. Modell trainieren

Jetzt trainieren wir 10 Epochen lang mit einer Batchgröße von 128 und halten 30% der Trainingsdaten für die Validierung zurück.

- Die Anzahl der **Epochen** gibt an, wie viele vollständige Durchläufe das Modell durch die Trainingsdaten macht. Normalerweise hörst du auf, bevor das Modell Overfitting zeigt oder sobald sich der Validierungsverlust nicht mehr verbessert.
- Die **Batchgröße** gibt an, wie viele Beobachtungen das Modell verarbeitet, bevor es seine Gewichte aktualisiert. Aus Gründen der Recheneffizienz ist sie meist eine Zweierpotenz.

In [ ]:
epochs = 10
accuracy_metrics = model.fit(
    xtrain, ytrain, batch_size=128, epochs=epochs, validation_split=0.3
)

Erklärung des Ergebnisses: Nach 10 Epochen erreicht das Modell etwa 99,3% Trainings-Accuracy und etwa 97,9% Validierungs-Accuracy. Dass die Validierungs-Accuracy leicht hinter der Trainings-Accuracy zurückbleibt, ist normal und ein leichtes Anzeichen dafür, dass das Modell begonnen hat, sich etwas enger an den Trainingsdatensatz anzupassen als an den Validierungsdatensatz.

### 4. Trainingskurven untersuchen

Die Darstellung von Accuracy und Verlust pro Epoche macht sichtbar, ob sich das Modell noch verbessert und ob es anfängt, Overfitting zu zeigen (die Trainingsmetrik verbessert sich weiter, während die Validierungsmetrik stagniert oder schlechter wird).

In [ ]:
from matplotlib import pyplot as plt

plt.plot(accuracy_metrics.history["accuracy"], label="train_accuracy")
plt.plot(accuracy_metrics.history["val_accuracy"], label="val_accuracy")
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.legend()

In [ ]:
print(
    f"At the end of the {epochs}th epoch the validation accuracy has reached {'{:.4f}'.format(accuracy_metrics.history['val_accuracy'][-1])}"
)

Erklärung des Ergebnisses: Die ausgegebene Validierungs-Accuracy in der letzten Epoche liegt bei etwa 0,9788, das Modell klassifiziert also ungefähr 98 von je 100 Ziffern richtig, die es noch nicht gesehen hat.

In [ ]:
from matplotlib import pyplot as plt

plt.plot(accuracy_metrics.history["loss"], label="train_loss")
plt.plot(accuracy_metrics.history["val_loss"], label="val_loss")
plt.xlabel("epochs")
plt.ylabel("loss")
plt.legend()

In [ ]:
print(
    f"At the end of the {epochs}th epoch the validation loss has decreased to {'{:.4f}'.format(accuracy_metrics.history['val_loss'][-1])}"
)

Erklärung des Ergebnisses: Der Validierungsverlust in der letzten Epoche liegt bei etwa 0,0930. Der Trainingsverlust sinkt weiter, und wenn der Validierungsverlust zu steigen beginnt, während der Trainingsverlust sinkt, ist diese Lücke das Erkennungszeichen von Overfitting.

#### Overfitting an den Kurven ablesen

![Beispiel für Overfitting](https://i.stack.imgur.com/FkPkr.png)

Overfitting lässt sich am besten am Validierungs**verlust** beurteilen statt an der Accuracy, weil die Accuracy nicht immer ein verlässliches Maß für die Leistung eines Klassifikators ist. Im Beispielplot oben steigt der Validierungsverlust mit fortschreitendem Training deutlich, während der Trainingsverlust weiter sinkt: ein klassisches Muster für Overfitting.

Sehen wir uns ein Bild aus dem Testdatensatz an und prüfen, was das Modell dafür vorhersagt.

In [ ]:
# Ein Bild aus dem Testdatensatz visualisieren
plt.imshow(xtest[8])

Das ist das per One-Hot Encoding kodierte tatsächliche Label für dieses Bild:

In [ ]:
# Ursprüngliches kodiertes Label zu den Testdaten
ytest[8]

In [ ]:
xtest.shape

Sag die Klassenwahrscheinlichkeiten für den ganzen Testdatensatz vorher:

In [ ]:
# Vorhersagen für deine Testdaten treffen
pred = model.predict(xtest)

In [ ]:
pred

`pred` enthält für jedes Bild in `xtest` die Wahrscheinlichkeit jeder der 10 Klassen.

In [ ]:
pred[8].argmax()

Erklärung des Ergebnisses: Die höchste Wahrscheinlichkeit liegt bei Index 5, das Modell sagt für `xtest[8]` also die Ziffer **5** vorher. Das tatsächliche Label unten ist ebenfalls 5, die Vorhersage ist also richtig.

In [ ]:
ytest_true[8]  # das tatsächliche Label für die Testdaten

In [ ]:
np.unique(ytrain)

### Confusion Matrix

Eine Confusion Matrix zeigt für jede tatsächliche Ziffer, wie sich die Vorhersagen des Modells auf alle Klassen verteilt haben. Die Diagonale zählt die richtigen Vorhersagen; die Zellen außerhalb der Diagonale zeigen, welche Ziffern miteinander verwechselt werden.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

plt.figure(figsize=(8, 8))
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true=ytest_true, y_pred=np.argmax(pred, axis=-1))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(ytest_true))
disp.plot()
plt.show()

Erklärung des Ergebnisses: Fast die gesamte Masse liegt auf der Diagonale, passend zur Validierungs-Accuracy von ~97,9%. Die wenigen Einträge außerhalb der Diagonale weisen auf die Ziffernpaare hin, die das Modell gelegentlich verwechselt (zum Beispiel 4 mit 9 oder 3 mit 5). Dort würden zusätzliches Training oder Regularisierung am meisten helfen.

### Aufgabe

- Variiere die Hyperparameter und führ das Modell erneut aus.
- Probier aus, die Anzahl der Filter, `kernel_size`, `padding`, `activation`, `pool_size`, `units` usw. zu ändern.
- Füg weitere `Conv2D`- und `Dense`-Schichten hinzu.
- Probier verschiedene Aktivierungsfunktionen in den `Conv2D`-Schichten aus.
- Probier beim Kompilieren des Modells verschiedene Optimierer aus.

### Bonus: Feature Maps visualisieren

Ein trainiertes CNN ist keine Blackbox: Wir können uns ansehen, worauf jeder Filter reagiert. Hier bauen wir ein kleines Modell neu auf, das die Aktivierungen direkt nach der ersten `Conv2D`-Schicht ausgibt, und stellen dann die 6 Feature Maps für eine Ziffer dar. Jede Feature Map hebt ein anderes elementares Muster hervor (zum Beispiel eine Kante in einer bestimmten Richtung).

In [ ]:
from tensorflow.keras.models import Model
from numpy import expand_dims

Alle Schichten im trainierten Modell:

In [ ]:
# Alle Schichten im erstellten Modell
model.layers

In [ ]:
model.inputs

Definiere das Modell neu, sodass es die Aktivierungen direkt nach der ersten verborgenen Schicht ausgibt:

In [ ]:
# Modell neu definieren, damit es die Ausgabe direkt nach der ersten verborgenen Schicht liefert
K.clear_session()
model_small = Model(inputs=model.inputs, outputs=model.layers[0].output)
model_small.summary()

Lade ein Bild in der benötigten Form:

In [ ]:
# Bild in der benötigten Form laden
img = xtrain[2]
plt.imshow(img.reshape(28, 28), cmap=plt.cm.Greys)

In [ ]:
img.shape

Erweitere die Dimensionen, sodass das Bild eine einzelne Beobachtung darstellt (Anzahl der Bilder, Breite, Höhe, Kanäle):

In [ ]:
# Dimensionen erweitern, sodass es eine einzelne 'Beobachtung' darstellt (Anzahl der Bilder,Breite,Höhe,Anzahl der Kanäle)
img = expand_dims(img, axis=0)
# img = expand_dims(img, axis=3)
img.shape

Hol die Feature Maps aus der ersten verborgenen Schicht:

In [ ]:
# Feature Map für die erste verborgene Schicht abrufen
feature_maps = model_small.predict(img)

Stell alle 6 Feature Maps dar. Jede ist die Ausgabe eines anderen Filters in der ersten Faltungsschicht:

In [ ]:
# Alle 6 Maps in 3*2 Quadraten darstellen
height = 3
width = 2
ix = 1
plt.figure(figsize=(12, 4))
for _ in range(height):
    for _ in range(width):
        # Subplot festlegen und Achse ausblenden
        ax = plt.subplot(height, width, ix)
        ax.set_xticks([])
        ax.set_yticks([])
        # Filterkanal in Graustufen darstellen
        plt.imshow(feature_maps[0, :, :, ix - 1], cmap="gray")
        ix += 1
# Abbildung anzeigen
plt.show()

Wenn du beim Training Overfitting beobachtest, probier es mit etwas Regularisierung:

- [BatchNormalization](https://keras.io/api/layers/normalization_layers/batch_normalization/): Füg zwischen CNN-Schichten eine Batch-Normalisierungsschicht ein.
- [Dropout](https://keras.io/api/layers/regularization_layers/dropout/): Füg zwischen dichten Schichten eine Dropout-Schicht ein.
- [Callbacks](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks): Hilfsfunktionen, die in bestimmten Phasen des Trainings ausgeführt werden. Sie können helfen, Overfitting zu verhindern, den Fortschritt zu visualisieren, Checkpoints zu speichern und Logs zu erzeugen. [EarlyStopping](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping) zum Beispiel beendet das Training, wenn sich eine überwachte Metrik nicht mehr verbessert: `EarlyStopping(monitor='val_loss', patience=5)`. Übergib den Callback über `model.fit(..., callbacks=[callback])`.

## Zusammenfassung

In diesem Notebook hast du:

- erklärt, wie Faltung, Stride, Padding und Pooling ein Bild transformieren und warum Filter Gewichte gemeinsam nutzen.
- in Keras ein kleines CNN mit zwei Blöcken aus Faltung und Pooling, einer `Flatten`- und einer `Dense`-Schicht mit Softmax erstellt (nur 8.790 Parameter).
- es auf MNIST trainiert, dabei etwa 97,9% Validierungs-Accuracy erreicht und die Accuracy- und Verlustkurven gelesen, um Overfitting einzuschätzen.
- das Modell mit einzelnen Vorhersagen, einer Confusion Matrix und einem Blick auf die Feature Maps der ersten Schicht evaluiert.

## Quellen & weiterführende Literatur

- [**Keras Conv2D layer**](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D): Die API der hier verwendeten Faltungsschicht.
- [**Keras Sequential model guide**](https://www.tensorflow.org/guide/keras/sequential_model): Wie sich Schichten zu einem Modell stapeln lassen.
- [**Image Kernels, explained visually**](https://setosa.io/ev/image-kernels/): Ein interaktiver Blick darauf, was ein Kernel mit einem Bild macht.
- [**Intuitively understanding convolutions for deep learning**](https://medium.com/data-science/intuitively-understanding-convolutions-for-deep-learning-1f6f42faee1): Eine visuelle Anleitung zu Faltungen, die bei der Intuition ansetzt.